# Drug Side Effects - Clasificacion y RAG con Mistral

Proyecto: Algoritmos de clasificacion aplicados a efectos secundarios de medicamentos.

Dataset: 5000 registros de pacientes con reacciones adversas a farmacos.

**Objetivos:**
1. Normalizar el dataset (valores inconsistentes, nulos, formatos mixtos)
2. Entrenar y seleccionar el mejor modelo de clasificacion para predecir severidad
3. Crear un agente RAG con Mistral AI para responder preguntas sobre el dataset

## 1. Instalacion de librerias

In [1]:
!pip install pandas numpy scikit-learn matplotlib seaborn 'mistralai==0.4.2' faiss-cpu transformers sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.1/133.1 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-sdk 0.4.2 requires orjson>=3.11.5, but you have orjson 3.10.18 which is incompatible.


## 2. Importaciones

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import io
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_auc_score)
from sklearn.feature_selection import SelectKBest, chi2

from mistralai.client import MistralClient
from mistralai.models.chat_completion import ChatMessage
import faiss
from sentence_transformers import SentenceTransformer
import torch

print("Todas las librerias importadas correctamente.")

Todas las librerias importadas correctamente.


## 3. Configuracion Mistral AI

Modelo recomendado: **mistral-large-latest** (2407) - mejor rendimiento para razonamiento complejo y RAG.
Para embeddings: **mistral-embed**.

In [4]:
from google.colab import userdata
api_key = userdata.get('Llavekey')

client = MistralClient(api_key=api_key)
MODEL = "mistral-large-latest"
EMBED_MODEL = "mistral-embed"

print(f"Mistral client configurado. Modelo: {MODEL}, Embeddings: {EMBED_MODEL}")

Mistral client configurado. Modelo: mistral-large-latest, Embeddings: mistral-embed


## 4. Carga del dataset

El archivo debe subirse a Colab con el nombre `drug_side_effects_5000.csv`.

In [6]:
# Cargar con encoding flexible
encodings = ["utf-8", "latin1", "cp1252", "iso-8859-1"]
df = None
for enc in encodings:
    try:
        df = pd.read_csv("drug_side_effects_5000.csv", encoding=enc)
        print(f"Cargado con encoding: {enc}")
        break
    except UnicodeDecodeError:
        continue

print(f"Shape: {df.shape}")
df.head()

Cargado con encoding: utf-8
Shape: (5000, 16)


,PatientID,Age,GENDER,Country,DrugName,Dosage (mg),sideEffect,Severity,Outcome,Report Date,TreatmentStart,ChronicCondition,Smoker,Alcohol Use,Hospitalized,Recovery Days
0,PT-101501,44,Female,Australia,Metformin,500mg,Diarrhea,Severe,Recovering,2023-03-21,2023-02-21,Diabetes,Yes,Unknown,No,18.0
1,PT-102586,54,Female,Germany,Ibuprofen,500,Stomach Pain,MILD,Recovered,11-27-2022,2022-10-11,Kidney Disease,Yes,Frequent,No,21 days
2,PT-102653,22,Female,UK,Metformin,500,Nausea,Mild,Recovered,2021-10-04,2021-08-08,diabetes,Yes,never,No,27d
3,PT-101055,74 years,Female,UK,Amoxicillin,20,Nausea,Mild,Recovered,06-23-2024,11/06/2024,Asthma,Yes,Frequent,no,10.0
4,PT-100705,50,Female,USA,Metformin,5,Abdominal Pain,Mild,Recovered,2023-12-30,2023-12-11,NaN,Yes,NaN,No,34 days


## 5. Diagnostico inicial

In [7]:
print("=== INFO ===")
df.info()
print("\n=== NULLS ===")
print(df.isnull().sum())
print("\n=== DTYPES ===")
print(df.dtypes)
print("\n=== HEAD ===")
df.head()

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   PatientID         5000 non-null   object
 1    Age              5000 non-null   object
 2   GENDER            5000 non-null   object
 3   Country           5000 non-null   object
 4   DrugName          5000 non-null   object
 5   Dosage (mg)       5000 non-null   object
 6   sideEffect        5000 non-null   object
 7    Severity         5000 non-null   object
 8   Outcome           5000 non-null   object
 9   Report Date       5000 non-null   object
 10  TreatmentStart    5000 non-null   object
 11  ChronicCondition  4355 non-null   object
 12  Smoker            5000 non-null   object
 13  Alcohol Use       3924 non-null   object
 14  Hospitalized      5000 non-null   object
 15  Recovery Days     4421 non-null   object
dtypes: object(16)
memory usage: 625.1+ KB

=== NULL

,PatientID,Age,GENDER,Country,DrugName,Dosage (mg),sideEffect,Severity,Outcome,Report Date,TreatmentStart,ChronicCondition,Smoker,Alcohol Use,Hospitalized,Recovery Days
0,PT-101501,44,Female,Australia,Metformin,500mg,Diarrhea,Severe,Recovering,2023-03-21,2023-02-21,Diabetes,Yes,Unknown,No,18.0
1,PT-102586,54,Female,Germany,Ibuprofen,500,Stomach Pain,MILD,Recovered,11-27-2022,2022-10-11,Kidney Disease,Yes,Frequent,No,21 days
2,PT-102653,22,Female,UK,Metformin,500,Nausea,Mild,Recovered,2021-10-04,2021-08-08,diabetes,Yes,never,No,27d
3,PT-101055,74 years,Female,UK,Amoxicillin,20,Nausea,Mild,Recovered,06-23-2024,11/06/2024,Asthma,Yes,Frequent,no,10.0
4,PT-100705,50,Female,USA,Metformin,5,Abdominal Pain,Mild,Recovered,2023-12-30,2023-12-11,NaN,Yes,NaN,No,34 days


## 6. Normalizacion del dataset

### 6.1 Renombrar columnas a snake_case

In [8]:
col_map = {
    "PatientID": "patient_id",
    " Age ": "age",
    "GENDER": "gender",
    "Country": "country",
    "DrugName": "drug_name",
    "Dosage (mg)": "dosage_mg",
    "sideEffect": "side_effect",
    " Severity ": "severity",
    "Outcome": "outcome",
    "Report Date": "report_date",
    "TreatmentStart": "treatment_start_date",
    "ChronicCondition": "chronic_condition",
    "Smoker": "smoker",
    "Alcohol Use": "alcohol_use",
    "Hospitalized": "hospitalized",
    "Recovery Days": "recovery_days",
}
df.rename(columns=col_map, inplace=True)
print(f"Columnas: {list(df.columns)}")

Columnas: ['patient_id', 'age', 'gender', 'country', 'drug_name', 'dosage_mg', 'side_effect', 'severity', 'outcome', 'report_date', 'treatment_start_date', 'chronic_condition', 'smoker', 'alcohol_use', 'hospitalized', 'recovery_days']


### 6.2 Conversion de tipos numericos

In [9]:
# Age: limpiar "years", "yo", espacios
def clean_age(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    val = val.replace("years", "").replace("year", "").replace("yo", "").replace("old", "").strip()
    try:
        return int(float(val))
    except:
        return np.nan

df["age"] = df["age"].apply(clean_age)

# Dosage: limpiar "mg"
def clean_dosage(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower().replace("mg", "").replace(" ", "")
    try:
        return int(float(val))
    except:
        return np.nan

df["dosage_mg"] = df["dosage_mg"].apply(clean_dosage)

# Recovery days: limpiar "days", "d"
def clean_recovery(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower().replace("days", "").replace("day", "").replace("d", "")
    try:
        return float(val)
    except:
        return np.nan

df["recovery_days"] = df["recovery_days"].apply(clean_recovery)

print(f"age: {df['age'].dtype}, min={df['age'].min()}, max={df['age'].max()}")
print(f"dosage_mg: {df['dosage_mg'].dtype}, min={df['dosage_mg'].min()}, max={df['dosage_mg'].max()}")
print(f"recovery_days: {df['recovery_days'].dtype}, min={df['recovery_days'].min()}, max={df['recovery_days'].max()}")

age: int64, min=18, max=90
dosage_mg: int64, min=5, max=500
recovery_days: float64, min=2.0, max=45.0


### 6.3 Conversion de fechas

In [10]:
def parse_date(val):
    if pd.isna(val):
        return pd.NaT
    val = str(val).strip()
    formats = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y", "%m/%d/%Y", "%d-%m-%Y", "%Y/%m/%d"]
    for fmt in formats:
        try:
            return pd.to_datetime(val, format=fmt)
        except:
            continue
    try:
        return pd.to_datetime(val, dayfirst=True)
    except:
        return pd.NaT

df["report_date"] = df["report_date"].apply(parse_date)
df["treatment_start_date"] = df["treatment_start_date"].apply(parse_date)

print(f"report_date: {df['report_date'].dtype}")
print(f"treatment_start_date: {df['treatment_start_date'].dtype}")

report_date: datetime64[ns]
treatment_start_date: datetime64[ns]


### 6.4 Limpieza de strings y estandarizacion

In [11]:
# Reemplazar strings que representan nulos
null_vals = ["nan", "NaN", "N/A", "NaT", "<NA>", "None", "none", "", "Unknown", "never"]

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace(null_vals, np.nan)
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

# Gender
def norm_gender(v):
    if pd.isna(v): return np.nan
    v = str(v).strip().lower()
    return {"male": "Male", "m": "Male", "female": "Female", "f": "Female"}.get(v, v)
df["gender"] = df["gender"].apply(norm_gender)

# Country
def norm_country(v):
    if pd.isna(v): return np.nan
    v = str(v).strip().title()
    return {"Usa": "USA", "U.S.A": "USA", "Uk": "UK", "U.K": "UK"}.get(v, v)
df["country"] = df["country"].apply(norm_country)

# Severity
def norm_severity(v):
    if pd.isna(v): return np.nan
    return {"mild": "Mild", "moderate": "Moderate", "severe": "Severe"}.get(str(v).strip().lower(), v)
df["severity"] = df["severity"].apply(norm_severity)

# Outcome
def norm_outcome(v):
    if pd.isna(v): return np.nan
    return {"recovered": "Recovered", "recovering": "Recovering",
            "fatal": "Fatal", "hospitalized": "Hospitalized"}.get(str(v).strip().lower(), v)
df["outcome"] = df["outcome"].apply(norm_outcome)

# Smoker
def norm_smoker(v):
    if pd.isna(v): return np.nan
    v = str(v).strip().lower()
    return {"yes": "Yes", "y": "Yes", "no": "No", "n": "No"}.get(v, v)
df["smoker"] = df["smoker"].apply(norm_smoker)

# Hospitalized
def norm_hosp(v):
    if pd.isna(v): return np.nan
    v = str(v).strip().lower()
    return {"yes": "Yes", "true": "Yes", "no": "No", "false": "No"}.get(v, v)
df["hospitalized"] = df["hospitalized"].apply(norm_hosp)

# Title case para drug_name y side_effect
df["drug_name"] = df["drug_name"].str.title().str.strip()
df["side_effect"] = df["side_effect"].str.title().str.strip()
df["chronic_condition"] = df["chronic_condition"].str.title().str.strip()

print("Strings limpiados y estandarizados.")

Strings limpiados y estandarizados.


### 6.5 Manejo de nulos

In [12]:
print("Nulos antes:")
print(df.isnull().sum())
print()

# Imputar numericos con mediana
df["age"] = df["age"].fillna(df["age"].median()).astype(int)
df["dosage_mg"] = df["dosage_mg"].fillna(df["dosage_mg"].median()).astype(int)
df["recovery_days"] = df["recovery_days"].fillna(df["recovery_days"].median())

# Imputar categoricos con moda
for col in ["chronic_condition", "alcohol_use"]:
    df[col] = df[col].fillna(df[col].mode()[0])

# Eliminar filas con fechas nulas
df.dropna(subset=["report_date", "treatment_start_date"], inplace=True)
df.reset_index(drop=True, inplace=True)

print("Nulos despues:")
print(df.isnull().sum())
print(f"\nShape final: {df.shape}")

Nulos antes:
patient_id                 0
age                        0
gender                     0
country                    0
drug_name                  0
dosage_mg                  0
side_effect                0
severity                   0
outcome                    0
report_date                0
treatment_start_date       0
chronic_condition        875
smoker                     0
alcohol_use             1673
hospitalized               0
recovery_days            579
dtype: int64

Nulos despues:
patient_id              0
age                     0
gender                  0
country                 0
drug_name               0
dosage_mg               0
side_effect             0
severity                0
outcome                 0
report_date             0
treatment_start_date    0
chronic_condition       0
smoker                  0
alcohol_use             0
hospitalized            0
recovery_days           0
dtype: int64

Shape final: (5000, 16)


## 10. Conclusiones

Pipeline completo:
1. Dataset desnormalizado con inconsistencias intencionales
2. Normalizacion: limpieza de columnas, tipos, fechas, strings y nulos
3. EDA: analisis exploratorio de variables
4. Clasificacion: 4 algoritmos comparados, mejor modelo seleccionado
5. RAG Agent: busqueda semantica con Mistral embeddings + generacion de respuestas

**Nota sobre Mistral:** Se recomienda `mistral-large-latest` para tareas de razonamiento
complejo y generacion RAG, y `mistral-embed` para embeddings. Es el modelo mas capaz
de Mistral actualmente, con buen equilibrio entre calidad y costo.